# 华为云命令触发摄像头截图 → AI识别 → 上传结果

**完整流程**:
```
IoTDA控制台下发 recognizeOnce 命令
        ↓
Python收到命令
        ↓
摄像头截取当前画面 → 本地保存
        ↓
AI模型识别（MushroomCNN 3分类）
        ↓
上传识别结果到华为云
        ↓
回复命令执行结果
```

先用下面第1步单独测试AI识别，确认模型加载和预测正常，再运行第2步完整集成。

---
## 第1步：单独测试 AI 模型识别截图

从 `captures/` 文件夹拿一张截图，送入你的 MushroomCNN 模型，输出识别结果。

In [ ]:
# ===== 第1步：单独测试 AI 模型识别截图 =====
import torch
import torch.nn as nn
import os
import cv2
import numpy as np
from torchvision import transforms
from PIL import Image

# ===================== 1. 配置 =====================
MODEL_PATH = r"D:\LHWYAN\download\environment\Python-AI\mushroom_cnn.pth"

# 3分类：必须和训练时的顺序一致
CLASS_NAMES = ["健康菌丝", "单一感染", "混合感染"]
CLASS_NAMES_EN = ["Healthy", "Single_Infected", "Mixed_Infected"]

IMG_SIZE = 100
DEVICE = torch.device("cpu")


# ===================== 2. 模型结构（和训练代码完全一致） =====================
class MushroomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 3)          # 3分类输出
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        out = self.classifier(x)
        return out


# ===================== 3. 加载模型 =====================
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"模型文件不存在：{MODEL_PATH}")

model = MushroomCNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
model.eval()

print(f"✅ 模型加载成功：{MODEL_PATH}")
print(f"📋 类别：{CLASS_NAMES}")


# ===================== 4. 图像预处理 =====================
predict_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# ===================== 5. 单张图片预测函数 =====================
def predict_mushroom_image(image_path):
    """
    输入一张图片路径，返回 (中文类别名, 置信度, 各类别概率列表)
    """
    image = Image.open(image_path).convert("RGB")
    image_tensor = predict_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(image_tensor)                # shape: [1, 3]
        probs = torch.softmax(output, dim=1)        # shape: [1, 3]
        confidence, pred_idx = torch.max(probs, dim=1)

        plant_status = CLASS_NAMES[pred_idx.item()]
        conf_value = confidence.item()
        all_probs = probs.squeeze().tolist()

    return plant_status, conf_value, all_probs


# ===================== 6. 测试：拿 captures 里最后一张截图测试 =====================
captures_dir = "captures"
if not os.path.exists(captures_dir):
    print("❌ captures 文件夹不存在，请先运行摄像头截图")
else:
    # 找最后一张截图
    jpg_files = [f for f in os.listdir(captures_dir) if f.endswith(".jpg")]
    if len(jpg_files) == 0:
        print("❌ captures 文件夹中没有截图，请先运行摄像头截图")
    else:
        jpg_files.sort(reverse=True)
        test_image = os.path.join(captures_dir, jpg_files[0])
        print(f"📷 使用截图：{test_image}\n")

        plant_status, conf, all_probs = predict_mushroom_image(test_image)

        print("=" * 50)
        print(f"🔍 识别结果：{plant_status}")
        print(f"📊 置信度：{conf:.4f} ({conf:.2%})")
        print("--- 各类别概率 ---")
        for i, (name, prob) in enumerate(zip(CLASS_NAMES, all_probs)):
            bar = "█" * int(prob * 30) + "░" * (30 - int(prob * 30))
            print(f"  {name:8s}  {prob:.2%}  {bar}")
        print("=" * 50)

        # 在 Notebook 中显示图片
        from IPython.display import display, Image as IPyImage
        display(IPyImage(filename=test_image))

---
## 第2步：完整集成代码（AI识别取代假数据）

运行这个单元格后，程序会：
1. 连接华为云 + 订阅命令Topic
2. 等待 `recognizeOnce` 命令
3. 收到命令 → **摄像头截图** → **AI模型识别** → **上传结果** → **回复命令**
4. 持续在线监听

In [ ]:
# =============================================================================
# 完整集成代码：华为云命令订阅 + 摄像头截图 + AI识别 + 属性上报
# =============================================================================

# ===== 配置区（根据实际情况修改） =====
CAMERA_URL = "http://192.168.43.98:8080/stream"
MODEL_PATH = r"D:\LHWYAN\download\environment\Python-AI\mushroom_cnn.pth"
CLASS_NAMES = ["健康菌丝", "单一感染", "混合感染"]
IMG_SIZE = 100

import json
import time
import ssl
import os
import cv2
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from paho.mqtt import client as mqtt_client
from paho.mqtt.client import CallbackAPIVersion


# ===================== 1. 华为云 IoTDA 连接参数 =====================
HOST = "412aa055e1.st1.iotda-device.cn-north-4.myhuaweicloud.com"
PORT = 8883
CLIENT_ID = "6a3a6e0d7f2e6c302f7e2ac9_myplantAIdevice_0_0_2026063012"
USERNAME = "6a3a6e0d7f2e6c302f7e2ac9_myplantAIdevice"
PASSWORD = "0d9d75fff0361c394d38d315223e49c4deebcf25f728af2130325c02198b1a0e"
DEVICE_ID = USERNAME
SERVICE_ID = "plantAI"
PUB_TOPIC = f"$oc/devices/{DEVICE_ID}/sys/properties/report"
CMD_SUB_TOPIC = f"$oc/devices/{DEVICE_ID}/sys/commands/#"


# ===================== 2. AI模型定义与加载（MushroomCNN 3分类） =====================
class MushroomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 3)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        out = self.classifier(x)
        return out

print("⏳ 加载AI模型...")
device = torch.device("cpu")
ai_model = MushroomCNN()
ai_model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
ai_model.to(device)
ai_model.eval()
predict_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
print(f"✅ AI模型加载成功：{MODEL_PATH}")


# ===================== 3. 摄像头截图函数 =====================
def capture_one_frame():
    os.makedirs("captures", exist_ok=True)
    cap = cv2.VideoCapture(CAMERA_URL)
    if not cap.isOpened():
        cap.release()
        return False, None, f"摄像头无法打开：{CAMERA_URL}"
    frame = None
    ret = False
    for _ in range(8):
        ret, frame = cap.read()
        if ret:
            break
        time.sleep(0.05)
    cap.release()
    if not ret or frame is None:
        return False, None, "摄像头画面读取失败"
    image_name = time.strftime("plant_%Y%m%d_%H%M%S.jpg")
    image_path = os.path.join("captures", image_name)
    cv2.imwrite(image_path, frame)
    return True, image_path, "截图成功"


# ===================== 4. AI识别函数 =====================
def ai_predict(image_path):
    """返回 (中文状态, 置信度)"""
    image = Image.open(image_path).convert("RGB")
    tensor = predict_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = ai_model(tensor)
        probs = torch.softmax(output, dim=1)
        confidence, pred_idx = torch.max(probs, dim=1)
    return CLASS_NAMES[pred_idx.item()], round(confidence.item(), 4)


# ===================== 5. 属性上报函数 =====================
def publish_plant_result(client, plant_status, confidence, camera_status="在线", ai_status="完成"):
    capture_time = time.strftime("%Y-%m-%d %H:%M:%S")
    payload = {
        "services": [
            {
                "service_id": SERVICE_ID,
                "properties": {
                    "plantStatus": plant_status,
                    "confidence": confidence,
                    "captureTime": capture_time,
                    "cameraStatus": camera_status,
                    "aiStatus": ai_status
                }
            }
        ]
    }
    msg = json.dumps(payload, ensure_ascii=False)
    result = client.publish(PUB_TOPIC, msg)
    if result.rc == 0:
        print(f"✅ 属性上报成功")
    else:
        print(f"❌ 属性上报失败，错误码：{result.rc}")


# ===================== 6. 回复命令结果 =====================
def send_command_response(client, request_id, result_code=0, result_name="success", result_msg=""):
    resp_topic = f"$oc/devices/{DEVICE_ID}/sys/commands/response/request_id={request_id}"
    response = {
        "result_code": result_code,
        "result_name": result_name,
        "result_msg": result_msg
    }
    client.publish(resp_topic, json.dumps(response))
    print(f"✅ 已回复命令执行结果（{result_name}）")


# ===================== 7. 命令处理回调（核心：截图 → AI识别 → 上传） =====================
def on_message(client, userdata, msg):
    topic = msg.topic
    payload_str = msg.payload.decode("utf-8", errors="ignore")
    print(f"\n📩 收到平台命令Topic：{topic}")
    print(f"📩 收到平台命令内容：{payload_str}")

    if "request_id=" not in topic:
        print("⚠️ 非命令消息，跳过")
        return
    request_id = topic.split("request_id=")[-1]

    try:
        cmd_data = json.loads(payload_str)
        command_name = cmd_data.get("command_name", "")
    except json.JSONDecodeError:
        print("❌ 命令内容JSON解析失败")
        return

    if command_name == "recognizeOnce":
        print("\n📸 收到 recognizeOnce 命令，开始截图...")

        # ----- 第1步：摄像头截图 -----
        ok, image_path, capture_msg = capture_one_frame()

        if not ok:
            print(f"❌ 摄像头截图失败：{capture_msg}")
            publish_plant_result(client, "未知", 0.0, "异常", "失败")
            send_command_response(client, request_id, 1, "capture failed", capture_msg)
            return

        print(f"✅ 摄像头截图成功：{image_path}")

        # ----- 第2步：AI模型识别 -----
        print(f"🧠 AI模型识别中...")
        try:
            plant_status, confidence = ai_predict(image_path)
            print(f"✅ AI识别完成：{plant_status} (置信度：{confidence:.4f})")
            ai_status = "完成"
        except Exception as e:
            print(f"❌ AI识别失败：{e}")
            plant_status = "未知"
            confidence = 0.0
            ai_status = "失败"

        # ----- 第3步：上传结果到华为云 -----
        publish_plant_result(client, plant_status, confidence, "在线", ai_status)

        # ----- 第4步：回复命令 -----
        send_command_response(client, request_id, 0, "success")

    else:
        print(f"⚠️ 未知命令：{command_name}，忽略")
        send_command_response(client, request_id, 1, "unknown command")


# ===================== 8. 连接MQTT =====================
def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        print("✅ MQTT连接成功")
        client.subscribe(CMD_SUB_TOPIC)
        print(f"✅ 已订阅命令主题：{CMD_SUB_TOPIC}")
    else:
        print(f"❌ MQTT连接失败，返回码：{reason_code}")

def on_disconnect(client, userdata, disconnect_flags, reason_code, properties):
    print(f"⚠️ MQTT连接断开，返回码：{reason_code}")

client = mqtt_client.Client(
    callback_api_version=CallbackAPIVersion.VERSION2,
    client_id=CLIENT_ID
)
client.username_pw_set(USERNAME, PASSWORD)
ssl_context = ssl.create_default_context()
client.tls_set_context(ssl_context)
client.on_connect = on_connect
client.on_disconnect = on_disconnect
client.on_message = on_message

client.connect(HOST, PORT, keepalive=60)
client.loop_start()
time.sleep(2)

print("\n" + "=" * 60)
print("🚀 程序已启动，等待华为云下发 recognizeOnce 命令...")
print(f"📷 摄像头：{CAMERA_URL}")
print(f"🧠 AI模型：{CLASS_NAMES}")
print(f"📂 截图保存：captures/ 文件夹")
print("=" * 60)


# ===================== 9. 保持运行 =====================
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🔌 用户中断，断开MQTT连接")
    client.loop_stop()
    client.disconnect()
    print("已断开与华为云IoT的连接")

---
## 第3步：测试方法

1. 先运行**第1步**单元格，确认模型能正确识别截图
2. 再运行**第2步**完整集成单元格（保持运行不要停）
3. 去华为云 IoTDA 控制台 → 设备 → 命令下发 → `plantAI` → `recognizeOnce` → 下发
4. 观察输出，应该看到：

```
📩 收到平台命令Topic：$oc/devices/xxx/sys/commands/request_id=xxx
📩 收到平台命令内容：{"service_id":"plantAI","command_name":"recognizeOnce","paras":{}}

📸 收到 recognizeOnce 命令，开始截图...
✅ 摄像头截图成功：captures/plant_20260630_XXXXXX.jpg
🧠 AI模型识别中...
✅ AI识别完成：健康菌丝 (置信度：0.9234)
✅ 属性上报成功
✅ 已回复命令执行结果（success）
```

**验证成功的3个标准**:
1. ✅ 本地 `captures/` 文件夹生成一张截图
2. ✅ AI模型返回正确的3分类结果（健康菌丝/单一感染/混合感染）
3. ✅ 华为云设备属性被刷新